
# Single-turn simulation (Phase 3)

Runs two-agent single-turn simulations using `SingleTurnSimulator`:
- seed agent A with a target emotion (synthetic templates aligned to DistilRoBERTa labels)
- agent B replies with a specified strategy prompt
- agent A follows up
- classify emotions before/after using DistilRoBERTa

Requires `OPENAI_API_KEY` in the environment (OpenAI Chat completions). Outputs: CSV and optional heatmap to `results/`.


In [ ]:
import getpass, os
os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter your OpenAI API key: ')


In [ ]:
# Optional: install/refresh package in hosted Colab
# !pip uninstall -y dynamic-conversation dynamic_conversation
# !pip cache purge
# !pip install --no-cache-dir git+https://github.com/Javin-Mendiratta/Dynamic-Conversation.git@derek_12_13

import os
from dynamic_conversation import SingleTurnSimulator, ResponseStrategy, compute_transition_confidence_intervals

# Ensure your OpenAI key is set. Uncomment and set directly if running interactively.
# # os.environ["OPENAI_API_KEY"] = "sk-..."


In [ ]:
# Full Phase 3 grid (baseline included). Increase runs_per_pair for real experiments.
emotions = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
strategies = [
    ResponseStrategy.VALIDATE,
    ResponseStrategy.EXPLORE,
    ResponseStrategy.REFRAME,
    ResponseStrategy.AFFIRM,
    ResponseStrategy.GUIDE,
    ResponseStrategy.NORMALIZE,
]

runs_per_pair = 1  # TODO: raise to 15-20 for report-ready stats
style_modifier = None  # e.g., 'empathetic and casual'
use_llm_seed = False
include_baseline = True

sim = SingleTurnSimulator(use_gpu=True)
df = sim.run_batch(
    emotions=emotions,
    strategies=strategies,
    runs_per_pair=runs_per_pair,
    style_modifier=style_modifier,
    use_llm_seed=use_llm_seed,
    include_baseline=include_baseline,
    save_metadata=True,
    save_csv='results/full_phase3_single_turn.csv',
    save_heatmap='results/full_phase3_single_turn_heatmap.png',
)

df.head()

# Display the saved heatmap inline (if available)
from IPython.display import Image, display
heatmap_path = 'results/full_phase3_single_turn_heatmap.png'
if Path(heatmap_path).exists():
    display(Image(filename=heatmap_path))
else:
    print(f"Heatmap not found at {heatmap_path}; ensure save_heatmap is set.")



Default grid runs all 7 emotions × 6 strategies + baseline. Adjust `runs_per_pair`, `style_modifier`, `use_llm_seed`, and output paths as needed. Keep outputs under `results/` and leave `save_metadata=True` (default) to log configs.



Use `compute_transition_confidence_intervals` to summarize shifts with CIs. For comparative analyses (e.g., strategy vs. baseline), rerun with higher `runs_per_pair` (15–20) and, if possible, another seed/style modifier.


In [ ]:
# Compute Wilson CIs for emotion distributions per (intended_emotion, strategy)
ci_df = compute_transition_confidence_intervals(df, confidence=0.95)
ci_df.head()


In [ ]:
# Inspect saved run metadata (config/seeds)
import json
from pathlib import Path
meta_path = Path("results/demo_single_turn.meta.json")
if meta_path.exists():
    print(meta_path.read_text())
else:
    print("Metadata file not found; ensure save_metadata=True in run_batch.")
